# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moriyamao/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Skill loaded: `hunting-leakage-and-validating` + `flyrank/flyrank-data`.*

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A — "What Predicts Health?" (Random Forest feature importance, ML Appendix, p.27).** Average Position comes out as the #1 predictor of Health Score at 43% importance, followed by Impressions (32%) and Scroll Depth (15%).

*My methodology question:* the paper itself is admirably upfront that "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal" — Health Score's own formula is Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll (20pts). Given that disclosure, my question is a follow-up rather than a challenge: **how much of that 43% is the model discovering something, versus the model re-deriving arithmetic that's already guaranteed by the formula?** The leakage skill's own test for this — train once WITH the suspect feature, once WITHOUT, and watch for a score collapse — would turn "descriptive rather than causal" into a quantified number instead of a caveat. As written, a reader can't tell whether removing Position would drop the model's fit by 5% or by 40%.

**Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, ML Appendix, p.29).** Content Age is reported as the strongest negative signal separating growing from declining pages, with Days Since Update and Days Visible as the strongest positive ones.

*My methodology question:* the label (`trend_direction` from 30d-vs-prev-30d impressions) is clear and not leakage-suspect on its face. What isn't stated on this page is the **split design** — the study spans 57 brands, and pages from the same brand plausibly share hidden character (template, publishing cadence, niche competitiveness). If the 71% holdout accuracy comes from a random row-level split rather than a brand-grouped split, some of that number could be the model learning "which brand is this" rather than "what predicts growth" in general — the same gap I found on my own model in Section 2 below. Since the paper doesn't disclose the split method here, I can't tell which case applies; I'd ask for that one line to be added rather than assume either way.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import os
if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    !git clone https://github.com/moriyamao/flyrank-ml-internship.git repo
    os.chdir('repo')

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

num_features = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'engagement_rate', 'scroll_rate',
                'ai_traffic_pct', 'avg_position', 'ctr', 'word_count', 'content_age_days',
                'days_since_last_update', 'search_volume', 'competition']
cat_features = ['content_type', 'main_intent', 'competition_level']
X = df[num_features + cat_features]
y = df['is_declining_label']
groups = df['client_id']

def fit_eval(train_idx, test_idx, label):
    pre = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), num_features),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='unknown')),
                           ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
    ])
    rf = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))])
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    proba = rf.predict_proba(X.iloc[test_idx])[:, 1]
    y_test = y.iloc[test_idx].values
    auc = roc_auc_score(y_test, proba)
    order = np.argsort(-proba)
    p50 = y_test[order[:50]].mean()
    p100 = y_test[order[:100]].mean()
    print(f'{label}: AUC={auc:.3f}  P@50={p50:.3f}  P@100={p100:.3f}  base_rate={y_test.mean():.3f}')
    return {'split': label, 'AUC': round(auc, 3), 'P@50': round(p50, 3), 'P@100': round(p100, 3), 'base_rate': round(y_test.mean(), 3)}

# BEFORE: random row-level split -- ignores that pages from the same client share hidden character.
rtr_idx, rte_idx = train_test_split(np.arange(len(df)), test_size=0.25, random_state=42, stratify=y)
before = fit_eval(rtr_idx, rte_idx, 'RANDOM split (before)')

# AFTER: grouped split by client_id -- the honest question, does it work on a client it never saw.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
gtr_idx, gte_idx = next(gss.split(X, y, groups))
after = fit_eval(gtr_idx, gte_idx, 'GROUPED split (after)')

random_overlap = len(set(groups.iloc[rtr_idx]) & set(groups.iloc[rte_idx]))
grouped_overlap = len(set(groups.iloc[gtr_idx]) & set(groups.iloc[gte_idx]))
print()
print('Clients shared between train/test -- random split:', random_overlap, '| grouped split:', grouped_overlap)

before_after = pd.DataFrame([before, after])
before_after

Cloning into 'repo'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 163 (delta 66), reused 95 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 1.87 MiB | 10.94 MiB/s, done.
Resolving deltas: 100% (66/66), done.
RANDOM split (before): AUC=0.745  P@50=0.960  P@100=0.890  base_rate=0.542
GROUPED split (after): AUC=0.600  P@50=0.560  P@100=0.560  base_rate=0.517

Clients shared between train/test -- random split: 31 | grouped split: 0


,split,AUC,P@50,P@100,base_rate
0,RANDOM split (before),0.745,0.96,0.89,0.542
1,GROUPED split (after),0.600,0.56,0.56,0.517


**The gap is the finding.** Random split AUC is **0.745**; grouped split AUC is **0.600** — a 0.145 drop. At precision@50 the gap is far starker: **0.96 under random split vs 0.56 under grouped split**, and the random split shares 31 clients between train and test while the grouped split shares zero by construction. That 0.96 number from Week 5's original random-style thinking would have been a wildly overstated claim — the model wasn't predicting decline in general, it was substantially recognizing *which client's pages these are* and reusing client-level regularities it saw in training. The grouped 0.600 is the honest number: a real but modest signal, the same one reported in Week 5.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# Attack checklist, applied to the Week-5 feature set:
suspect_columns = {'trend_direction', 'trend_pct'}          # label-derived
product_flags = {'provider_used', 'model_used'}              # decision-derived, never features
final_features = set(num_features) | set(cat_features)

print('Label-derived columns present in final features:', suspect_columns & final_features)
print('Product-flag columns present in final features:  ', product_flags & final_features)
print('IDs present in final features (should be empty):  ', {'client_id', 'content_id'} & final_features)

Label-derived columns present in final features: set()
Product-flag columns present in final features:   set()
IDs present in final features (should be empty):   set()


In [3]:
# Prove the test harness actually catches leakage: deliberately inject a label-derived
# feature (trend_pct, the same number the label is computed FROM) and watch the score jump.
def eval_feature_set(feat_num, label):
    Xf = df[feat_num + cat_features]
    train_idx, test_idx = next(gss.split(Xf, y, groups))
    pre = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), feat_num),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='unknown')),
                           ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
    ])
    rf = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))])
    rf.fit(Xf.iloc[train_idx], y.iloc[train_idx])
    proba = rf.predict_proba(Xf.iloc[test_idx])[:, 1]
    auc = roc_auc_score(y.iloc[test_idx], proba)
    print(f'{label}: AUC={auc:.3f}')
    return auc

auc_honest = eval_feature_set(num_features, 'WITHOUT suspect (final honest features)')
auc_leaky = eval_feature_set(num_features + ['trend_pct'], 'WITH suspect (trend_pct injected on purpose)')
print()
print('Jump when the suspect feature is added:', round(auc_leaky - auc_honest, 3))

WITHOUT suspect (final honest features): AUC=0.600
WITH suspect (trend_pct injected on purpose): AUC=1.000

Jump when the suspect feature is added: 0.4


**Confession confirmed.** Injecting `trend_pct` — the exact column the label is computed from — pushes AUC from 0.600 to a perfect **1.000**. That's the textbook symptom the skill describes ("a collapse from ~1.0 to ~0.7 is the confession", read in reverse: the jump TO ~1.0 is the same confession). This confirms two things at once: (1) my test harness genuinely detects leakage when it's present, so the honest 0.600 number in Section 2 isn't honest by accident — I checked; and (2) `trend_pct` and `trend_direction` are correctly excluded from the real feature set, exactly as flagged in Week 4's data contract and Week 5's method choice.

**Full checklist:**
- [x] Timeline: every feature (impressions, clicks, position, CTR, word count, etc.) is a trailing-90-day snapshot value, knowable before any 30-day trend label is computed — no feature is built from a window that contains the label window.
- [x] No label-derived columns in the final features (confirmed above; the deliberate-injection test also shows what it would look like if there were).
- [x] No product flags as features — `provider_used`/`model_used` are excluded per the data dictionary.
- [x] Population selection: no filtering on anything from the outcome window (the full 30,000-row starter slice is used, not a subset defined by later performance).
- [x] Split grouped by `client_id` (Section 2).
- [x] Base rate printed next to every metric (Section 2 table).
- [x] Top feature importance sanity-checked in Week 5 (`impressions_90d`, `avg_position`, `content_age_days` — none suspiciously dominant at ~0.21 max).
- [x] Metrics computed out-of-fold on the held-out client split, never in-sample.
- [ ] Sealed/holdout claim: not applicable — this notebook doesn't claim a sealed evaluation; the starter CSV is a single snapshot, not a panel with a sealed final month.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence from Week 5** (Section 3 interpretation): *"the model shows a directional edge over the baseline specifically for tight, high-precision top-K review queues."*

That sentence was already written in fairly careful language, but after this audit it's worth tightening further — it was based on the grouped-split number, which is good, but it didn't disclose that a naively-validated version of the same model would have looked dramatically stronger (AUC 0.745, P@50 0.96) for reasons that have nothing to do with real predictive skill. Leaving that gap unstated is itself a way a claim can go further than the evidence, even if every individual number quoted was accurate.

**Rewrite:** *"Under a client-grouped validation split — the honest test of whether this generalizes to clients the model never saw — the Random Forest model shows a small, directional improvement over the Week-4 rule baseline at tight review-queue sizes (K≤50), and is roughly comparable to or weaker than the baseline at broader queue sizes (K≥100). This is decision-support signal, not a strong or causal claim: a naive random-row split on the same data and features produces a far larger apparent score (AUC 0.745 vs 0.600) that this audit attributes to client memorization rather than genuine predictive skill, which is why the grouped number, not the random one, is the one that should be trusted or reported."*

The rewrite adds: (1) the split method as a load-bearing qualifier, not a footnote, (2) an explicit comparison to the inflated number so a reader can see the size of the risk being guarded against, and (3) careful words throughout — "small," "directional," "decision-support," "attributes to," never "proves" or "causes."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run it yourself once to confirm**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.